In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os
import re
import gc
import sys
import shutil
import subprocess
import unicodedata
import pkg_resources
import pandas as pd
import numpy as np
import random
from tqdm.auto import tqdm
from fractions import Fraction
from threading import Thread
from joblib import Parallel, delayed
import sacrebleu
import ctranslate2

gc.collect()
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,garbage_collection_threshold:0.6"
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# 1) Python version & Device
print("Python:", sys.version.replace("\n", " "))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA Available:", torch.cuda.is_available())
print("CPU Available:", os.cpu_count())
print("CTranslate2 支持的计算类型:", ctranslate2.get_supported_compute_types("cuda"))
print("CT2 Version:", ctranslate2.__version__)

# 2) All installed packages
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
# packages = sorted(pkg_resources.working_set, key=lambda x: x.key)
# for pkg in packages:
#     print(f"{pkg.key}=={pkg.version}")

# 3) Config
DATA_PATH = '/kaggle/input/deep-past-initiative-machine-translation'
OUTPUT_PATH = '/kaggle/working'
CT2_BASE_PATH = os.path.join(OUTPUT_PATH, 'ct2_models')
os.makedirs(CT2_BASE_PATH, exist_ok=True)
DEBUG = False

# DATA
TRAIN_DF_PATH = os.path.join(DATA_PATH, 'train.csv')
TEST_DF_PATH = os.path.join(DATA_PATH, 'test.csv')

# Inference
MODEL_SPECS = [
    {
        "name": "Model A",
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0323-data1-all1404/GuoqinGu/byt5-xl-0323-data1-all-checkpoint-1404",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-a-byt5-xl-0323-data1/transformers/all-1404/1/pre_convert_model"
    },
    {    
        "name": "Model B", 
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0323-data1-llmlabel-all2424/GuoqinGu/byt5-xl-0323-data1_llmlabel-all-checkpoint-2424",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-b-byt5-xl-0323-data1-llmlabel/transformers/all-2424/1/pre_convert_model"
    },
    {
        "name": "Model C",
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0321-data2-all1452/GuoqinGu/byt5-xl-0321-data2-all-checkpoint-1452", 
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-c-byt5-xl-0321-data2/transformers/all-1452/1/pre_convert_model"
    },
    {
        "name": "Model D",
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0321-data2-llmlabel-all2472/GuoqinGu/byt5-xl-0321-data2_llmlabel-all-checkpoint-2472",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-d-byt5-xl-0321-data2-llmlabel/transformers/all-2472/1/pre_convert_model"
    },
    {
        "name": "Model E",
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0321-data3-llmlabel-all2622/GuoqinGu/byt5-xl-0321-data3_llmlabel-all-checkpoint-2622",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-e-byt5-xl-0321-data3-llmlabel/transformers/all-2622/1/pre_convert_model"
    },
    {    
        "name": "Model F", 
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0321-data3-all1602/GuoqinGu/byt5-xl-0321-data3-all-checkpoint-1602",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-f-byt5-xl-0321-data3/transformers/all-1602/1/pre_convert_model"
    },

    {
        "name": "Model G",
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0323-data2-llmlabel-synth12-all3159/GuoqinGu/byt5-xl-0323-data2_llmlabel_synth12-all-checkpoint-3159",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-g-byt5-xl-0323-data2-llmlabel-synth12/transformers/all-3159/1/pre_convert_model"
    },
    {    
        "name": "Model H", 
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0323-data3-llmlabel-synth12-all3309/GuoqinGu/byt5-xl-0323-data3_llmlabel_synth12-all-checkpoint-3309",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-h-byt5-xl-0323-data3-llmlabel-synth12/transformers/all-3309/1/pre_convert_model"
    },

    {
        "name": "Model I",
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0322-data2-split-checkpoint-1308",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-i-byt5-xl-0322-data2/transformers/split-1308/1/pre_convert_model"
    },
    {    
        "name": "Model J", 
        "ckpt_path": "/kaggle/input/datasets/guoqingu/byt5-xl-0322-data3-split-checkpoint-1443",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-j-byt5-xl-0322-data3/transformers/split-1443/1/pre_convert_model"
    },

    {    
        "name": "Model K", 
        "ckpt_path": "/kaggle/input/models/hongori/byt-5-weighted-bs48-epoch4/transformers/default/1/checkpoint-2512",
        "ct2_path": "/kaggle/input/models/guoqingu/ct2-model-k-byt5-xl-weighted/transformers/bs48-ep4/1/pre_convert_model"
    },
]

MAX_LENGTH = 1024
QUANTIZATION = "int8_float32"
# ===== 单模型推理参数 =====
BATCH_SIZE = 10
NUM_BEAMS = 8
NUM_BEAM_CANDS = 4
LENGTH_PENALTY = 1.05
EARLY_STOPPING = True
REPETITION_PENALTY = 1.0

# ===== 多温度 sampling =====
SAMPLE_TEMPERATURES = [0.60, 0.80, 1.05]
NUM_SAMPLE_PER_TEMP = 2
MBR_TOP_P = 0.92

MAX_NEW_TOKENS = 512
MBR_POOL_CAP = 200

# ===== chrF++混合 BLEU、Jaccard 和长度奖励 =====
MBR_W_CHRF = 0.55
MBR_W_BLEU = 0.25
MBR_W_JACCARD = 0.20
MBR_W_LENGTH = 0.10

FALLBACK_TEXT = "The tablet is too damaged to translate."


/tmp/ipykernel_41/316340458.py:12: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CUDA Available: True
CPU Available: 4
CTranslate2 支持的计算类型: {'float32', 'int8_float32', 'int8_float16', 'float16', 'int8'}
CT2 Version: 4.7.1
pandas: 2.2.2
numpy: 2.0.2
torch: 2.8.0+cu126
transformers: 4.57.1


In [2]:
# 模型在推理阶段按统一列表顺序加载。
# 这里不预先常驻加载模型，避免后续扩展到更多模型时主流程分散。
model = None
tokenizer = None


In [3]:
def process_fractions_dynamic(text_to_process):
    if not isinstance(text_to_process, str):
        return str(text_to_process)
    # Step0: 预处理
    # --- 0.1 清理数值边界的特殊标记符号 ---
    # (123) [123] ˹123˺ ⌈123⌋ 「123」 → 123
    text_to_process = re.sub(r'\((\d+)\)', r'\1', text_to_process)
    text_to_process = re.sub(r'\[(\d+)\]', r'\1', text_to_process)
    text_to_process = re.sub(r'˹(\d+)˺', r'\1', text_to_process)
    text_to_process = re.sub(r'⌈(\d+)⌋', r'\1', text_to_process)
    text_to_process = re.sub(r'「(\d+)」', r'\1', text_to_process)

    # --- 0.2 转换 X+X 格式的数值：循环处理直到没有更多加法运算需要处理 ---
    def calculate_addition(match):
        try:
            val1 = float(match.group(1))
            val2 = float(match.group(2))
            result = val1 + val2
            # 格式化逻辑: 如果是整数则转int str，否则保留有效小数
            return "{:.5f}".format(result).rstrip('0').rstrip('.') if result % 1 != 0 else str(int(result))
        except ValueError:
            return match.group(0)
    addition_pattern = r'(\d+(?:\.\d+)?)\s*\+\s*(\d+(?:\.\d+)?)'
    while True:
        new_text = re.sub(addition_pattern, calculate_addition, text_to_process)
        if new_text == text_to_process:
            break
        text_to_process = new_text

    # --- 0.3 月份替换 ---
    # Month I - month XII -> month 1 - month 12
    roman_to_arabic = {
        "I": "1", "II": "2", "III": "3", "IV": "4", "V": "5", "VI": "6",
        "VII": "7", "VIII": "8", "IX": "9", "X": "10", "XI": "11", "XII": "12"
    }
    roman_pattern = r'\b(Month|month)\s+(VIII|XII|VII|III|XI|IX|VI|IV|II|X|V|I)\b'
    def replace_roman(match):
        return f"{match.group(1)} {roman_to_arabic[match.group(2)]}"
    text_to_process = re.sub(roman_pattern, replace_roman, text_to_process)

    # Step1: 转换 X / X 格式的分数
    # --- 1.1 分数转换：规则内的分数转换到 Unicode 分数符号 ---
    fraction_map = {
        r"1\s*/\s*6": "⅙",
        r"1\s*/\s*4": "¼",
        r"1\s*/\s*3": "⅓",
        r"1\s*/\s*2": "½",
        r"2\s*/\s*3": "⅔",
        r"3\s*/\s*4": "¾",
        r"5\s*/\s*6": "⅚",
        # r"1\s*/\s*8": "⅛", r"3\s*/\s*8": "⅜", r"5\s*/\s*8": "⅝", r"7\s*/\s*8": "⅞",
    }
    for pattern, char in fraction_map.items():
        text_to_process = re.sub(pattern, char, text_to_process)

    # --- 1.2 通用数值分数转换：剩余的分数形式转换为小数格式 ---
    def calc_fraction(match):
        groups = match.groups()
        if groups[0]: # 如果匹配到了整数部分 (例如 "5 11/12")
            integer_part = int(groups[0])
            num = int(groups[1])
            den = int(groups[2])
            val = integer_part + (num / den)
        else: # 只匹配到了分数部分 (例如 "11/12")
            num = int(groups[3])
            den = int(groups[4])
            val = num / den
        # 格式化为5位小数 (例如 5.916666... -> 5.91667)
        return "{:.5f}".format(val).rstrip('0').rstrip('.') if val % 1 != 0 else str(int(val))
    # 模式1 (混合分数): (\d+)\s+(\d+)/(\d+) -> 匹配 "5 11/12"  |  模式2 (纯分数): (\d+)/(\d+) -> 匹配 "11/12"
    fraction_pattern = r'(\d+)\s+(\d+)/(\d+)|(\d+)/(\d+)'   
    text_to_process = re.sub(fraction_pattern, calc_fraction, text_to_process)
    
    # Step2: 转换小数格式的分数到 Unicode 分数符号
    # --- 2.1 处理长浮点数损坏数值 ---
    # 正则逻辑：匹配 数字 + 小数点 + 至少5位数字
    # \d+\.\d{5,} 匹配如 1.3333300000000001
    # lambda 表达式将其截断至小数点后 5 位
    text_to_process = re.sub(r'(\d+\.\d{5})\d+', r'\1', text_to_process)

    # --- 2.2 统一成 UNICODE 分数符号 ---
    fraction_lookup = {
            (1, 6): "⅙", # .166
            (1, 4): "¼", # .25
            (1, 3): "⅓", # .333
            (1, 2): "½", # .5
            (2, 3): "⅔", # .666
            (3, 4): "¾", # .75
            (5, 6): "⅚", # .833
        #    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
        #    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞"
    }
    def replacer(match):
        full_str = match.group(0)
        try:
            val = float(full_str)
            integer_part = int(val)
            decimal_part = val - integer_part
            # Return integer if decimal is negligible
            if decimal_part < 0.0001: 
                return str(integer_part)
            # Limit denominator handles 0.3333 -> 1/3
            frac = Fraction(decimal_part).limit_denominator(12) 

            unicode_frac = fraction_lookup.get((frac.numerator, frac.denominator))
            if unicode_frac:
                if integer_part == 0:
                    return unicode_frac
                else:
                    # Current logic: Space between Integer and Fraction
                    return f"{integer_part} {unicode_frac}"

            return full_str 
        except ValueError:
            return full_str
    processed_text = re.sub(r'\b\d+\.\d+\b', replacer, text_to_process)
    
    # Step3: Robustness check
    # --- 3.1 保证带分数中间有空格 ---
    processed_text = re.sub(r'(\d)([¼½¾⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞])', r'\1 \2', processed_text)
    
    return processed_text

def preprocess_transliteration(text):
    if pd.isna(text):
        return ""
    text = str(text)
    
    # --- 1. 字符替换与括号处理 ---
    char_map = str.maketrans({
        "X": "x",
        "ā": "a",
        "Ş": "Ṣ",
        "ş": "ṣ",
        "Ș": "Ṣ",
        "ș": "ṣ",
        "Ț": "Ṭ",
        "ț": "ṭ",
        "İ": "I",
        "Ī": "I",
        "ī": "i",
        "Î": "I",
        "î": "i",
        "ı": "i",
        "h": "ḫ",
        "H": "Ḫ",
        "ḥ": "ḫ",
        "Ḥ": "Ḫ",
        "=": "-",
        "(": "{",  # for deterministic
        ")": "}",
        "—": "-",
        "–": "-",
    })
    text = text.translate(char_map)
    text = text.replace("ᵈ", r"{d}").replace("ᵏⁱ", r"{ki}")   # 手动操作
    text = text.replace(r"{{", "{").replace(r"}}", "}")
    text = text.replace("lá+lá", "lá-lá").replace("la+la", "la-la")
    text = text.replace("{?}", "").replace("{!}", "").replace("-/", "-")
    # Remove bad characters
    upper_chars = ["⁰","¹","²","³","⁴","⁵","⁶","⁷","⁸","⁹","⁻","°", "˚"]
    for ch in upper_chars:
        text = text.replace(ch, "")
    bad_chars = """"ʺʾ’´ʼˊ˘ˋˇˈʿʹ'`^˹˺⸢⸣⌜⌝「」⌈⌋⌊⌉˻˼⸤⸥≪≫《》«»⟨⟩˃‹›⁽⁾ˁˀ"""
    for ch in bad_chars:
        text = text.replace(ch, "")
    bad_chars = """!?⸮ʔ⸮,/;˷˴|:+⁺*"""
    for ch in bad_chars:
        text = text.replace(ch, " ")
    
    # --- 2. 角标统一处理 ---
    # 4 a-bi4-a 0.5 2+10 ù a-šùr-DU10: 4 a-bi₄-a 0.5 2+10 ù a-šùr-DU₁₀
    # 定义允许的字母范围（包含普通字母和特殊字符）
    letters_str = "a-zA-ZšŠṣṢṭṬḫḪ"
    # 定义数字到角标的映射表
    subscript_trans = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    # 正则逻辑：匹配紧跟在字母后面的数字
    def replace_group_with_subscript(match):
        digits = match.group(0)
        return digits.translate(subscript_trans)
    text = re.sub(f"(?<=[{letters_str}])\d+", replace_group_with_subscript, text)

    # --- 3. Gap替换 ---
    # ['PN'/'Pn'] -> '<gap>'
    pattern = r'\bPN\b|\bPn\b'
    text = re.sub(pattern, '<gap>', text)
    # break/broken -> ''
    text = re.sub(r'\bbreak\b|\bbroken\b', '', text)
    # [] -> [xxx]
    text = re.sub(r'\[[\sx?]*\]', '[xxx]', text)
    # Multi-dot patterns → <gap>
    text = re.sub(r'(?:\.\s+){2,}\.', lambda m: m.group(0).replace(' ', ''), text)
    text = re.sub(r'\.{3,}(\s+\.{3,})*', '<gap>', text)
    text = re.sub(r'……|…', '<gap>', text)
    # x patterns → <gap>
    text = re.sub(r'xx+', '<gap>', text)
    # 处理孤立的 x (前后是空格或边界)
    text = re.sub(r'\bx\b', '<gap>', text)
    # 直接替换 {large break}
    text = text.replace("{large break}", "<gap>")
    # 正则替换 {N broken lines} (例如 {3 broken lines}, {4 broken lines})
    text = re.sub(r'\{\d+\s+broken\s+lines\}', '<gap>', text)
    # Temporarily protect tokens from cleanup
    text = text.replace("<gap>", "\x00GAP\x00")
    # Remove bad characters
    bad_chars = "<>[]"
    for ch in bad_chars:
        text = text.replace(ch, "")
    # Restore tokens
    text = text.replace("\x00GAP\x00", " <gap> ")
    text = text.replace("{ <gap> }", "<gap>")
    text = text.replace("[ <gap> ]", "<gap>")
    
    # Normalize whitespace and trim
    text = re.sub(r"\s+", " ", text).strip().strip("-")
    text = text.replace(" -", "-").replace("- ", "-")
    text = text.replace("> .", ">.").replace("> :", ">:").replace("> ’", ">’")

    # merge <gap>
    pattern = r'<gap>([ \-\t]*<gap>)+'
    text = re.sub(pattern, '<gap>', text)
    
    return text

<>:181: SyntaxWarning: invalid escape sequence '\d'
<>:181: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_41/58297003.py:181: SyntaxWarning: invalid escape sequence '\d'
  text = re.sub(f"(?<=[{letters_str}])\d+", replace_group_with_subscript, text)


In [4]:
def postprocess_test_translation(text):
    # Return empty string for invalid outputs
    if not isinstance(text, str) or not text.strip():
        return ""

    # Normalize specific diacritics
    processed_text = text.replace("ḫ", "h").replace("Ḫ", "H")

    processed_text = unicodedata.normalize("NFKC", text)
    processed_text = text.lower()

    # Vowels
    processed_text = re.sub(r'[āâ]', 'a', text)
    processed_text = re.sub(r'[ēê]', 'e', text)
    processed_text = re.sub(r'[īî]', 'i', text)
    processed_text = re.sub(r'[ōô]', 'o', text)
    processed_text = re.sub(r'[ūû]', 'u', text)

    # Consonants
    processed_text = re.sub(r'[ṣ]', 's', text)
    processed_text = re.sub(r'[ṭ]', 't', text)
    processed_text = re.sub(r'[ḍ]', 'd', text)
    processed_text = re.sub(r'[ḫ]', 'h', text)
    processed_text = re.sub(r'[š]', 'sh', text)

    # Determinatives / glosses
    processed_text = re.sub(r'\{[^}]+\}', '', text)
    processed_text = re.sub(r'\([^)]*\)', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    # Convert subscript digits to normal digits
    sub_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    processed_text = processed_text.translate(sub_map)

    # Normalize gap markers
    processed_text = re.sub(r"(\[x\]|\(x\)|\bx\b)", "<gap>", processed_text, flags=re.I)
    processed_text = re.sub(r"(\.{3,}|…|\[\.+\])", "<gap>", processed_text)

    # Merge adjacent gaps
    processed_text = re.sub(r"<gap>\s*<gap>", " <gap> ", processed_text)
    processed_text = re.sub(r"<big_gap>\s*<big_gap>", " <gap> ", processed_text)

    # Remove some parenthetical morphology notes
    processed_text = re.sub(
        r"\((fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)",
        "",
        processed_text,
        flags=re.I,
    )

    # Temporarily protect tokens from cleanup
    processed_text = processed_text.replace("<gap>", "\x00GAP\x00")

    # Remove bad characters
    bad_chars = '!?()"—–<>⌈⌋⌊[]+ʾ/;'
    processed_text = processed_text.translate(str.maketrans("", "", bad_chars))

    # Restore tokens
    processed_text = processed_text.replace("\x00GAP\x00", " <gap> ")

    # Handle common fractions from decimals
    frac_map = {
        r"\.5\b": " ½",
        r"\.25\b": " ¼",
        r"\.75\b": " ¾",
        r"\.33+\d*\b": " ⅓",
        r"\.66+\d*\b": " ⅔",
    }
    for pattern, replacement in frac_map.items():
        processed_text = re.sub(r"(\d+)" + pattern, r"\1" + replacement, processed_text)
        processed_text = re.sub(r"\b0" + pattern, replacement.strip(), processed_text)

    # Remove repeated single words
    processed_text = re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", processed_text)

    # Remove repeated n-grams
    for n in range(4, 1, -1):
        pattern = r"\b((?:\w+\s+){" + str(n - 1) + r"}\w+)(?:\s+\1\b)+"
        processed_text = re.sub(pattern, r"\1", processed_text)

    # Normalize whitespace and trim
    processed_text = re.sub(r"\s+", " ", processed_text).strip().strip("-")

    return processed_text

In [5]:
def get_token_limits(transliteration_len):
    # 根据你的拟合结果填入系数
    m, c = 1.1427, -1.1173
    up, down = 200, 200
    
    max_tokens = int(m * transliteration_len + (c + up))
    min_tokens = max(0, int(m * transliteration_len + (c - down)))
    
    return min_tokens, max_tokens

class AkkadDataset(Dataset):
    def __init__(self, df):
        self.ids = df['id'].tolist()
        self.texts = ["Translate Akkadian to English: " + str(t) for t in df['transliteration']]
        self.min_tokens = df['min_t'].tolist()
        self.max_tokens = df['max_t'].tolist()

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        return {
            "id": self.ids[i],
            "text": self.texts[i],
            "min_t": self.min_tokens[i],
            "max_t": self.max_tokens[i]
        }

def collate_fn(batch):
    ids = [x["id"] for x in batch]
    texts = [x["text"] for x in batch]
    batch_min = int(np.min([x["min_t"] for x in batch]))
    batch_max = int(np.max([x["max_t"] for x in batch]))
    
    inputs = tokenizer(texts, max_length=MAX_LENGTH, padding=True, truncation=True, return_tensors="pt")
    
    return ids, inputs, batch_min, batch_max

In [6]:
# 版本1：双模型加权 MBR
# 保持现有候选生成方式和加权 MBR

class MBRSelector:
    # pairwise score = chrF++ + BLEU + Jaccard 的加权组合
    # final score    = pairwise score + length bonus
    def __init__(
        self,
        pool_cap=32,
        w_chrf=0.55,
        w_bleu=0.25,
        w_jaccard=0.20,
        w_length=0.10,
    ):
        self._chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
        self._bleu_metric = sacrebleu.metrics.BLEU(effective_order=True)
        self.pool_cap = pool_cap
        self.w_chrf = w_chrf
        self.w_bleu = w_bleu
        self.w_jaccard = w_jaccard
        self.w_length = w_length
        self._pw_total = max(w_chrf + w_bleu + w_jaccard, 1e-9)

    def _chrfpp(self, a, b):
        if not a or not b:
            return 0.0
        return float(self._chrf_metric.sentence_score(a, [b]).score)

    def _bleu(self, a, b):
        if not a or not b:
            return 0.0
        try:
            return float(self._bleu_metric.sentence_score(a, [b]).score)
        except Exception:
            return 0.0

    @staticmethod
    def _jaccard(a, b):
        ta, tb = set(a.lower().split()), set(b.lower().split())
        if not ta and not tb:
            return 100.0
        if not ta or not tb:
            return 0.0
        return 100.0 * len(ta & tb) / len(ta | tb)

    def _pairwise_score(self, a, b):
        score = (
            self.w_chrf * self._chrfpp(a, b)
            + self.w_bleu * self._bleu(a, b)
            + self.w_jaccard * self._jaccard(a, b)
        )
        return score / self._pw_total

    @staticmethod
    def _length_bonus(lengths, idx):
        if len(lengths) == 0:
            return 100.0
        median = float(np.median(lengths))
        sigma = max(median * 0.4, 5.0)
        z = (lengths[idx] - median) / sigma
        return 100.0 * float(np.exp(-0.5 * z * z))

    @staticmethod
    def _dedup(xs):
        seen = set()
        out = []
        for x in xs:
            x = str(x).strip()
            if x and x not in seen:
                out.append(x)
                seen.add(x)
        return out

    def pick(self, candidates):
        cands = self._dedup(candidates)
        if self.pool_cap:
            cands = cands[:self.pool_cap]

        n = len(cands)
        if n == 0:
            return ""
        if n == 1:
            return cands[0]

        lengths = [len(c.split()) for c in cands]
        scores = []

        for i in range(n):
            pairwise = sum(
                self._pairwise_score(cands[i], cands[j])
                for j in range(n)
                if j != i
            ) / max(1, n - 1)

            length_bonus = self._length_bonus(lengths, i)
            total = pairwise + self.w_length * length_bonus
            scores.append(total)

        return cands[int(np.argmax(scores))]


In [7]:
def ensure_ct2_model(ckpt_path, ct2_path):
    """确保模型已经被编译为 CT2 格式，如果没有则进行转换"""
    if not os.path.exists(ct2_path):
        print(f"\n⚙️ 正在转换模型到 CTranslate2 ({QUANTIZATION})...\n  源: {ckpt_path}\n  目标: {ct2_path}")
        result = subprocess.run(
            [
                "ct2-transformers-converter",
                "--model", ckpt_path,
                "--output_dir", ct2_path,
                "--quantization", QUANTIZATION,
                "--force",
            ],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print("STDERR:", result.stderr)
            raise RuntimeError("模型转换失败，请检查上方输出")
        print(f"✅ {ct2_path} 转换完成")
    else:
        print(f"✅ {ct2_path} 已存在，跳过转换")

def load_ct2_model_on_device(model_spec, device_index):
    """在指定 GPU 索引上加载 CTranslate2 Translator"""
    return {
        "name": model_spec["name"],
        "translator": ctranslate2.Translator(
            model_spec["ct2_path"],
            device="cuda",
            device_index=[device_index], # 分配到具体的 GPU 卡
            compute_type=QUANTIZATION,
            inter_threads=1,
            flash_attention=False
        ),
        "tokenizer": AutoTokenizer.from_pretrained(model_spec["ckpt_path"]),
        "device_index": device_index
    }

def release_model_bundle(model_bundle):
    if model_bundle is None: return
    if "translator" in model_bundle:
        # 卸载 CTranslate2 模型以释放显存
        del model_bundle["translator"]
    if "tokenizer" in model_bundle: 
        del model_bundle["tokenizer"]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [8]:
def build_ct2_collate_fn(current_tokenizer):
    def _collate(batch):
        ids = [x["id"] for x in batch]
        texts = [x["text"] for x in batch]
        batch_min = int(np.min([x["min_t"] for x in batch]))
        batch_max = int(np.max([x["max_t"] for x in batch]))
        
        # 对于 CT2，我们需要传入 Token String 的列表，而不是 Tensor ID
        inputs = current_tokenizer(texts, truncation=True, max_length=MAX_LENGTH)
        source_tokens = [current_tokenizer.convert_ids_to_tokens(seq) for seq in inputs.input_ids]
        
        return ids, source_tokens, batch_min, batch_max
    return _collate

def _decode_ct2_tokens(tokens, tokenizer):
    return tokenizer.decode(
        tokenizer.convert_tokens_to_ids(tokens),
        skip_special_tokens=True,
    )

def generate_candidate_pools_ct2(source_tokens, translator, tokenizer):
    batch_size = len(source_tokens)

    # ---- Beam Search 候选 ----
    beam_results = translator.translate_batch(
        source_tokens,
        beam_size=NUM_BEAMS,
        num_hypotheses=NUM_BEAM_CANDS,
        length_penalty=LENGTH_PENALTY,
        repetition_penalty=REPETITION_PENALTY,
        max_decoding_length=MAX_NEW_TOKENS,
        max_input_length=MAX_LENGTH,
    )
    beam_texts = [
        [_decode_ct2_tokens(hyp, tokenizer) for hyp in r.hypotheses]
        for r in beam_results
    ]

    # ---- 多温度 Nucleus Sampling 候选 ----
    all_sample_texts = [[] for _ in range(batch_size)]
    for temperature in SAMPLE_TEMPERATURES:
        for _ in range(NUM_SAMPLE_PER_TEMP):
            sample_results = translator.translate_batch(
                source_tokens,
                beam_size=1,
                num_hypotheses=1,
                sampling_topk=0,            
                sampling_topp=MBR_TOP_P,    
                sampling_temperature=temperature,
                repetition_penalty=REPETITION_PENALTY,
                max_decoding_length=MAX_NEW_TOKENS,
                max_input_length=MAX_LENGTH,
            )
            for i, result in enumerate(sample_results):
                all_sample_texts[i].append(_decode_ct2_tokens(result.hypotheses[0], tokenizer))

    # ---- 合并候选池 ----
    pools = [beam_texts[i] + all_sample_texts[i] for i in range(batch_size)]
    return pools

def run_ct2_inference_worker(loader, model_bundle, results_dict):
    device_index = model_bundle["device_index"]
    translator = model_bundle["translator"]
    tokenizer = model_bundle["tokenizer"]
    pools_by_id = {}
    
    # CT2 是在 C++ 层执行推理，不需要 torch.inference_mode()
    for ids, source_tokens, min_t, max_t in tqdm(loader, desc=f"{model_bundle['name']} on cuda:{device_index}"):
        batch_pools = generate_candidate_pools_ct2(source_tokens, translator, tokenizer)
        for sample_id, pool in zip(ids, batch_pools):
            pools_by_id[sample_id] = pool
            
    results_dict[f"cuda:{device_index}"] = pools_by_id

In [9]:
# ================= 数据准备阶段 =================
if DEBUG:
    test_df = pd.read_csv(TRAIN_DF_PATH)
    test_df = test_df.rename(columns={'oare_id': 'id', 'translation': 'translation_groundtruth'})
else:
    test_df = pd.read_csv(TEST_DF_PATH)

test_df["transliteration"] = test_df["transliteration"].apply(process_fractions_dynamic)
test_df["transliteration"] = test_df["transliteration"].apply(preprocess_transliteration)
test_df['transliteration_char_length'] = test_df.apply(lambda row: len(row['transliteration']), axis=1)
test_df = test_df.sort_values(by='transliteration_char_length', ascending=False).reset_index(drop=True)
test_df[['min_t', 'max_t']] = test_df.apply(
    lambda row: get_token_limits(row['transliteration_char_length']),
    axis=1, result_type='expand'
)
sample_ids = test_df['id'].tolist()

# 奇偶交叉 (Interleave)，平衡双卡的 Padding 负载
df_0 = test_df.iloc[0::2].reset_index(drop=True)
df_1 = test_df.iloc[1::2].reset_index(drop=True)

dataset_0 = AkkadDataset(df_0)
dataset_1 = AkkadDataset(df_1)

mbr_selector = MBRSelector(
    pool_cap=MBR_POOL_CAP,
    w_chrf=MBR_W_CHRF,
    w_bleu=MBR_W_BLEU,
    w_jaccard=MBR_W_JACCARD,
    w_length=MBR_W_LENGTH,
)

all_model_pools = {spec["name"]: {} for spec in MODEL_SPECS}

# ================= 模型推理阶段 =================
for model_spec in MODEL_SPECS:
    print(f"\n{'='*50}\n处理 {model_spec['name']}")
    
    # 1. 确保模型已被转换为 CT2 格式
    ensure_ct2_model(model_spec["ckpt_path"], model_spec["ct2_path"])
    
    # 2. 在 0卡 和 1卡 分别加载 CT2 Translator
    bundle_0 = load_ct2_model_on_device(model_spec, 0) # device_index = 0
    bundle_1 = load_ct2_model_on_device(model_spec, 1) # device_index = 1

    # 3. 构建返回 token 列表的 DataLoader
    loader_0 = DataLoader(dataset_0, batch_size=BATCH_SIZE, shuffle=False, collate_fn=build_ct2_collate_fn(bundle_0["tokenizer"]))
    loader_1 = DataLoader(dataset_1, batch_size=BATCH_SIZE, shuffle=False, collate_fn=build_ct2_collate_fn(bundle_1["tokenizer"]))

    results_dict = {}

    # 4. 开启双线程并行推理
    t0 = Thread(target=run_ct2_inference_worker, args=(loader_0, bundle_0, results_dict))
    t1 = Thread(target=run_ct2_inference_worker, args=(loader_1, bundle_1, results_dict))

    t0.start()
    t1.start()
    t0.join()
    t1.join()

    # 5. 聚合双卡结果
    all_model_pools[model_spec["name"]].update(results_dict.get("cuda:0", {}))
    all_model_pools[model_spec["name"]].update(results_dict.get("cuda:1", {}))

    # 6. 清理内存 / 显存
    release_model_bundle(bundle_0)
    release_model_bundle(bundle_1)
    
    # 【可选】如果你发现 kaggle/working 空间 (20GB) 不够用，可以开启以下代码，推理完一个就删掉一个转换后的文件
    # shutil.rmtree(model_spec["ct2_path"], ignore_errors=True)
    # print(f"🧹 已清理 {model_spec['ct2_path']} 以释放磁盘空间")

# ================= MBR 聚合与提交阶段 =================
print("\nStarting MBR selection...")
# 单条样本的处理逻辑
def process_single_sample(sample_id):
    raw_pool = []
    # 从外层作用域读取 all_model_pools 和 MODEL_SPECS
    for model_spec in MODEL_SPECS:
        raw_pool.extend(all_model_pools[model_spec["name"]].get(sample_id, []))

    cleaned_pool = [postprocess_test_translation(candidate) for candidate in raw_pool]
    cleaned_pool = [candidate for candidate in cleaned_pool if candidate]

    # 调用全局的 mbr_selector
    chosen = mbr_selector.pick(cleaned_pool)

    if not chosen:
        fallback = postprocess_test_translation(raw_pool[0]) if raw_pool else ""
        chosen = fallback if fallback else FALLBACK_TEXT

    return (sample_id, chosen)

# 使用 joblib 开启 4 核并行计算
results = Parallel(n_jobs=4)(
    delayed(process_single_sample)(sample_id)
    for sample_id in tqdm(sample_ids, desc="MBR Pooling")
)
submission = pd.DataFrame(results, columns=['id', 'translation'])
submission = submission.sort_values(by='id', ascending=True).reset_index(drop=True)
submission.to_csv(os.path.join(OUTPUT_PATH, 'submission.csv'), index=False, encoding='utf-8')

print(submission.shape)
print(submission.head())
print("✅ Saved submission.csv")


处理 Model A
✅ /kaggle/input/models/guoqingu/ct2-model-a-byt5-xl-0323-data1/transformers/all-1404/1/pre_convert_model 已存在，跳过转换


Model A on cuda:0:   0%|          | 0/1 [00:00<?, ?it/s]

Model A on cuda:1:   0%|          | 0/1 [00:00<?, ?it/s]


处理 Model B
✅ /kaggle/input/models/guoqingu/ct2-model-b-byt5-xl-0323-data1-llmlabel/transformers/all-2424/1/pre_convert_model 已存在，跳过转换


Model B on cuda:0:   0%|          | 0/1 [00:00<?, ?it/s]

...

Model K on cuda:0:   0%|          | 0/1 [00:00<?, ?it/s]

Model K on cuda:1:   0%|          | 0/1 [00:00<?, ?it/s]


Starting MBR selection...


MBR Pooling:   0%|          | 0/4 [00:00<?, ?it/s]

(4, 2)
   id  \
0   0   
1   1   
2   2   
3   3   

                                                                                                                                                                                                                                                                                                            translation  
0                                                                                                                                                                    Thus the Kanesh colony: Speak to the <gap> of the decree, our messenger, every single colony, and the trading stations: A tablet came to the City.  
1                                                                                                                                                              In accordance with the tablet of the city , from this day on, whoever buys meteoric iron from the profit <gap> or <gap> the kārum of Kaneš will take it.  
2  As